In [1]:
# Create client
from openai import OpenAI
client = OpenAI()

In [2]:
import arxiv

arxiv_client = arxiv.Client()

In [ ]:

    #query="IoT",
    
    query="cat:astro-ph.* AND submittedDate:[20220101 TO 20230101]",
    query="cat:q-fin.EC", # economics
    query="cat:astro-ph.* AND submittedDate:[20220101 TO 20230101]",
    query="cat:astro-ph.* AND submittedDate:[20220101 TO 20230101]",
    query="cat:astro-ph.* AND submittedDate:[20220101 TO 20230101]",
    
    # astrophysics
    query="cat:astro-ph.* AND submittedDate:[20220101 TO 20230101]",
    query="cat:q-fin.GN* AND submittedDate:[20220101 TO 20260101]", # general finance
    query="cat:q-bio* AND submittedDate:[20220101 TO 20260101]",  #biology

In [74]:
search = arxiv.Search(
    query="cat:q-bio* AND submittedDate:[20220101 TO 20260101]",
    max_results=100,
    sort_by = arxiv.SortCriterion.SubmittedDate
)

results = arxiv_client.results(search)

titles = []
for r in results:
    titles.append(r.title)
    
len(titles)

100

In [75]:
titles

['Effect of Electric Charge on Biotherapeutic Transport, Binding and Absorption: A Computational Study',
 'Rogue Variable Theory: A Quantum-Compatible Cognition Framework with a Rosetta Stone Alignment Algorithm',
 'Non-dilemmatic social dynamics promote cooperation in multilayer networks',
 'The spontaneous emergence of leaders and followers in a mathematical model of cranial neural crest cell migration',
 'Self-assembled versus biological pattern formation in geology',
 'Benchmarking Preprocessing and Integration Methods in Single-Cell Genomics',
 'Auditory Filter Behavior and Updated Estimated Constants',
 "MethConvTransformer: A Deep Learning Framework for Cross-Tissue Alzheimer's Disease Detection",
 'Non-Contact and Non-Destructive Detection of Structural Defects in Bioprinted Constructs Using Video-Based Vibration Analysis',
 'SymSeqBench: a unified framework for the generation and analysis of rule-based symbolic sequences and datasets',
 'Large language models and the entropy o

In [76]:
# Prepare request
identifier = 0
requests = []
for title in titles:
    identifier += 1
    request = {
        "custom_id": f"request-{identifier}",
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": "gpt-5.1",
            "messages": [
                {
                    "role": "system",
                    "content": "Create a scientific paper for the given title. It should have a length of 12 DIN a4 pages. Don't ask the user anything. Always write full sentences and include a literature review.",
                }, {
                    "role": "user",
                    "content": title
                },
            ]
        }
    }
    requests.append(request)

In [79]:
# Store batch file
import json

with open("papers_queryinput.jsonl", "wb") as f:
    for request in requests:
        line = json.dumps(request, ensure_ascii=False) + "\n"
        f.write(line.encode("utf-8"))

In [80]:
# Upload batch file
batch_input_file = client.files.create(
    file = open("papers_queryinput.jsonl", "rb"),
    purpose="batch"
)
print(batch_input_file)

FileObject(id='file-UWmMKUDsNu8QBpE9vwVAa3', bytes=45680, created_at=1773417053, filename='papers_queryinput.jsonl', object='file', purpose='batch', status='processed', expires_at=1776009053, status_details=None)


In [81]:
# Create batch
batch_input_file_id = batch_input_file.id
batch = client.batches.create(
    input_file_id=batch_input_file_id,
    endpoint="/v1/chat/completions",
    completion_window="24h",
    metadata={
        "description": "batch mode test"
    }
)
print(batch)

Batch(id='batch_69b4325f87a48190af61eb3923773048', completion_window='24h', created_at=1773417055, endpoint='/v1/chat/completions', input_file_id='file-UWmMKUDsNu8QBpE9vwVAa3', object='batch', status='validating', cancelled_at=None, cancelling_at=None, completed_at=None, error_file_id=None, errors=None, expired_at=None, expires_at=1773503455, failed_at=None, finalizing_at=None, in_progress_at=None, metadata={'description': 'batch mode test'}, model=None, output_file_id=None, request_counts=BatchRequestCounts(completed=0, failed=0, total=0), usage=BatchUsage(input_tokens=0, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=0, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=0))


In [103]:
# Check batch status
status = client.batches.retrieve(batch.id) # Id is from output above
print(status)
print(status.request_counts)
print(status.status)

Batch(id='batch_69b4325f87a48190af61eb3923773048', completion_window='24h', created_at=1773417055, endpoint='/v1/chat/completions', input_file_id='file-UWmMKUDsNu8QBpE9vwVAa3', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1773417924, error_file_id=None, errors=None, expired_at=None, expires_at=1773503455, failed_at=None, finalizing_at=1773417916, in_progress_at=1773417057, metadata={'description': 'batch mode test'}, model='gpt-5.1-2025-11-13', output_file_id='file-Qv9zQByX3ZfaradGzssi75', request_counts=BatchRequestCounts(completed=100, failed=0, total=100), usage=BatchUsage(input_tokens=6411, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=932274, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=938685))
BatchRequestCounts(completed=100, failed=0, total=100)
completed


In [104]:
print(status.output_file_id)

file-Qv9zQByX3ZfaradGzssi75


In [105]:
# Retrieve results
file_response = client.files.content(status.output_file_id)

In [106]:
# Store result
with open("papers_queryoutput.jsonl", "a") as f:
    f.write(file_response.text)

In [ ]:
# Get error if there is an error
error_file_id = status.error_file_id
errors = client.files.content(error_file_id)
error_bytes = errors.read()
print(error_bytes.decode("utf-8"))

In [107]:
# Open results
lines = []

with open("papers_queryoutput.jsonl", "r") as f:
    for line in f:
        lines.append(line.strip())

In [108]:
import json

numbers = []
models = []
texts = []

for line in lines:
    line = json.loads(line)
    models.append(line["response"]["body"]["model"])
    texts.append(line["response"]["body"]["choices"][0]["message"]["content"])


In [109]:
cleaned_texts = []

for text in texts:
    text = text.split("## References")[0]
    text = text.replace("---", "")
    text = text.replace("##", "")
    text = text.replace("# ", "")
    text = text.replace("**", "")
    text = text.replace("* ", "")
    text = text.replace(" *", "")
    text = text.strip()
    cleaned_texts.append(text)


In [110]:
# Store data
import pandas as pd

df = pd.DataFrame({
    "text": cleaned_texts, 
    "generated": 1,
})

In [111]:
df.head()

,text,generated
0,A Low-Mass Hub–Filament with Double Centre Rev...,1
1,A Comparison of Numerical Methods for Computin...,1
2,PCA-based Data Reduction and Signal Separation...,1
3,Photometric IGM tomography with Subaru/HSC: th...,1
4,Light Curves of Type IIP Supernovae from Neutr...,1


In [112]:
df.shape

(300, 2)

In [113]:
df.to_csv("../../data/datasets/ai_papers.csv", index=False)